#### Functions from Assignment 0

In [2]:
import import_ipynb
import numpy as np
from random import randint, shuffle
from math import sqrt
from sympy import factorint
from sympy import Poly , symbols , GF , invert
#pip install numpy --upgrade
#np.__version__
#previous installed version: 1.26.4
#current installed version: 2.2.4
#some changes with respect to previous version (release 2.0.0):
# - The default integer type on Windows is now int64 rather than int32, matching the behavior on other platforms,
# - The maximum number of array dimensions is changed from 32 to 64

In [3]:
# Finds the m- ary expansion of n
def getExpansion (n ,m):
    listOfDigits =[]
    while n >= m:
        digit =n%m
        listOfDigits . append ( digit )
        n =(n - digit ) // m
    listOfDigits.append(n)
    return listOfDigits
    
def intToText (n):
    t= getExpansion(n ,256)
    myString =''
    for i in t :
        myString = myString + chr(i)
    return myString

def textToInt (s):
    n =0
    k =0
    for i in s :
        n=n +ord( i) *(256** k)
        k=k +1
    return n

# Output the multiplicative inverse of a modulo p
def multInverse (a , p):
    result = extendedGCD (a , p)
    if result [0]!=1: # Error message if a and p are not relatively prime
        s=" Numbers needs to be relatively prime "
        return s
    inv = result [1]% p
    return inv

#extended euclidean algorithm
# Output [r,s,t] satisfying s*a+t*b=r=gcd(a,b)
def extendedGCD(a , b):
    r0 , r=a ,b
    s0 , s =1 ,0
    t0 , t =0 ,1
    while (r >0):
        tempr , temps , tempt =r ,s ,t
        q= r0 // r
        r ,s , t=r0 - q*r ,s0 -q*s ,t0 - q*t
        r0 , s0 , t0 = tempr , temps , tempt
    return [r0 ,s0 , t0 ]

def fast2Power (a ,n ,m):
    res = 1
    while n > 0:
        if n % 2 == 1: #If the bit is 1 multiply by the corresponding square
            res = ( res * a ) % m
        a =( a * a) % m
        n = n // 2
    return res

### Functions for Assignment 3

In [5]:
def findSquareRoot (N ,p) :
    N0 = N % p
    if fast2Power (N0, (p - 1) // 2 ,p) == 1: #Euler ’s criterion
        if p % 4 == 3:
            x1 = fast2Power (N0, (p + 1) // 4 , p) #see assignment 2 exercise 2 theoretical part
            y1 = p - x1
            return [x1 , y1]
        else :
            for i in range(1 , (( p - 1) // 2) + 1):
                if (i * i) % p == N0:
                    x1 = i
                    y1 = p - i
                    return [x1 , y1]
    return []

def generateCurve (E , p):
    if isElliptic (E , p) == False :
        print (" This is not an elliptic curve ")
        return None
    A, B = E
    listOfPoints =["O"]
    for x in range (p):
        a =(x**3 + A*x + B) % p
        if a == 0:
            listOfPoints.append ([x ,0])
        if fast2Power (a, (p - 1) // 2, p) == 1: # Euler ’s criterion there are solutions
            y1, y2 = findSquareRoot(a, p)
            listOfPoints.append([x, y1])
            listOfPoints.append([x, y2])
    return listOfPoints

# Part I (common part)

## Implementation part

### Exercise 1

In [707]:
def isElliptic (E ,p ):
    A, B = E
    discr = (4*( A **3) +27*( B **2) )%p
    return discr != 0

def pointOnCurve (P ,E ,p) :
    if P == "O":
        return True
    else :
        A, B = E
        x, y = P
        return (y **2) %p ==( x **3+ A* x+B) %p

In [715]:
E = [0,7] #represents the elliptic curve y^2 = x^3 + 7
p = 2**256 - 2**32 - 977
x = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798
y = 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8
P = [x, y]
if not isElliptic(E,p):
    print("E is not elliptic!")
elif pointOnCurve(P, E, p):
    print("P is on the curve!")
else:
    print("P is not on the curve!")

P is on the curve!


### Exercise 2

In [718]:
E = [5,12]
p = 13
C = generateCurve(E, p)
C

['O', [0, 5], [0, 8], [2, 2], [2, 11], [7, 0], [10, 3], [10, 10]]

In [720]:
len(C) <= p + 1 + 2*sqrt(p) and len(C) >= p + 1 - 2*sqrt(p)

True

### Exercise 3

#### (a)

In [722]:
def addPoints(P,Q,E,p):
    A = E[0]
    B = E[1]
    if P == "O":
        return Q
    elif Q == "O":
        return P
    x1, y1 = P
    x2, y2 = Q
    if x1 == x2 % p and y1 == -y2 % p:
        return "O"
    else:
        if P != Q:
            lmbda = (y2 - y1) * multInverse(x2 - x1, p) % p
        else:
            lmbda = (3 * fast2Power(x1, 2, p) + A) * multInverse(2*y1, p) % p
        x3 = (fast2Power(lmbda, 2, p) - x1 - x2) % p
        y3 = (lmbda * (x1 - x3) - y1) % p
        return [x3, y3]

#### (b)

In [730]:
E = [5, 12]
p = 13
P1, Q1 = "O", "O"
P2, Q2 = "O", [0,5]
P3, Q3 = [2, 2], "O"
P4, Q4 = [10, 3], [10, -3]
P5, Q5 = [2, 2], [2, 11]
P6, Q6 = [10, 10], [0, 5]
P7, Q7 = [10, 3], [7, 0]
P8, Q8 = [10, 3], [10, 3]
P9, Q9 = [0, 5], [2, 11]
Plist = [P1, P2, P3, P4, P5, P6, P7, P8, P9]
Qlist = [Q1, Q2, Q3, Q4, Q5, Q6, Q7, Q8, Q9]
for i in range(len(Plist)):
    sum = addPoints(Plist[i], Qlist[i], E, p)
    if pointOnCurve(sum, E, p):
        print(i + 1, ":", Plist[i], "+", Qlist[i], "=", sum)

1 : O + O = O
2 : O + [0, 5] = [0, 5]
3 : [2, 2] + O = [2, 2]
4 : [10, 3] + [10, -3] = O
5 : [2, 2] + [2, 11] = O
6 : [10, 10] + [0, 5] = [0, 8]
7 : [10, 3] + [7, 0] = [10, 10]
8 : [10, 3] + [10, 3] = [7, 0]
9 : [0, 5] + [2, 11] = [7, 0]


### Exercise 4

#### (a)

In [21]:
def doubleAndAdd(P, n, E, p):
    res = "O"
    while n > 0:
        if n % 2 == 1:
            res = addPoints(res, P, E, p)
        P = addPoints(P, P, E, p)
        n = n // 2
    return res    

#### (b)

In [732]:
E = [0,7] #represents the elliptic curve y^2 = x^3 + 7
p = 2**256 - 2**32 - 977
x = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798
y = 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8
G = [x, y]
n = 165717357988647532
nG = doubleAndAdd(G, n, E, p)
print("nG =\n", nG)

nG =
 [51524656361346136203439460631008348936841590752868015863048885409956560359198, 79203802285035089814150287439488171030915061256999835886514121114625556531371]


### Exercise 5

### (a)

In [736]:
def addPoints(P,Q,E,N):
    A = E[0]
    B = E[1]
    if P == "O":
        return Q
    elif Q == "O":
        return P
    x1, x2 = P[0], Q[0]
    y1, y2 = P[1], Q[1]
    if x1 == x2 % N and y1 == -y2 % N:
        return "O"
    else:
        if P != Q:
            d1 = extendedGCD(x2 - x1, N)[0]
            if d1 != 1:
                return [-1, d1]
            lmbda = (y2 - y1) * multInverse(x2 - x1, N) % N
        else:
            d2 = extendedGCD(2 * y1, N)[0]
            if d2 != 1:
                return [-1, d2]
            lmbda = (3 * fast2Power(x1, 2, N) + A) * multInverse(2 * y1, N) % N
        x3 = (fast2Power(lmbda, 2, N) - x1 - x2) % N
        y3 = (lmbda * (x1 - x3) - y1) % N
        return [x3, y3]
        
def doubleAndAdd(P, n, E, p):
    res = "O"
    while n > 0:
        if n % 2 == 1:
            res = addPoints(res, P, E, p)
        P = addPoints(P, P, E, p)
        n = n // 2
    return res    

In [738]:
def generateRand(N):
    A, x, y = randint(0, N - 1), randint(0, N - 1), randint(0, N - 1)
    return [A, x, y]
    
def factorLenstra(N, boundary = 5000000):
    """
    Tries to find out non trivial divisors of N in n attempts.
    N: the number to be factorised N = pq, p, q primes,
    boundary: the number of attempts
    return: 1 if we gave up or a positive integer if we succeeded.
    """
    A, x, y = generateRand(N)
    B = (fast2Power(y, 2, N) - fast2Power(x, 3, N) - A * x) % N
    P = [x, y]
    d = 1
    i = 2
    while i < boundary:
        if (4 * fast2Power(A, 3, N) + 27 * fast2Power(B, 2, N)) % N == 0:
            A, x, y = generateRand(N)
            i += 1
        E = [A, B]
        P =  doubleAndAdd(P, i, E, N)
        if P == 'O':
            A, x, y = generateRand(N)
            i +=1
        x1, y1 = P
        if x1 == -1:
            d = y1
            break
        i +=1
    return d

#### (b)

In [741]:
N = 516083
p = factorLenstra(N)
q = N // p
print(N, "=", p, "*", q)
N == p*q

516083 = 907 * 569


True

In [743]:
N = 1833779393
p = factorLenstra(N)
q = N // p
print(N, "=", p, "*", q)
N == p*q

1833779393 = 12569 * 145897


True

In [745]:
N = 13487843290022369713
while True:
    p = factorLenstra(N, 1200)
    q = N // p
    if p != 1 and q != 1:
        break
print(N, "=", p, "*", q)
N == p*q

13487843290022369713 = 6228182549 * 2165614637


True

### Exercise 6

In [747]:
# How to encode a message to a point
def messageToPoint(m, E, p, K = 100) :
    #K is the error tollerance
    # Message m needs to satisfy (m +1)K<p
    if (m +1) *K >= p:
        print (" Error tolerance or size of message space need to be changed !")
        return None
    for j in range (0, K - 1):
        x = K * m + j
        z = (x ** 3 + E [0] * x + E [1]) % p
        # print (x,z)
        if fast2Power (z, (p - 1) // 2, p) == 1:
            y = fast2Power (z ,(p + 1) // 4 , p)
            M = [x ,y]
            # print (P)
            return M
    print ("No encoding found , increase K")
    return None

#### (a)

In [750]:
def mSign(x, p):
    if x % p > (p - 1) // 2:
        return 1
    else:
        return 0

def yCoordinate(x, b, E, p):
    A, B = E
    a = (x**3 + A*x + B) % p
    y = findSquareRoot(a, p)[0]
    signY = mSign(y, p)
    if b == 0:
         if signY == 0:
             return y
         else:
             return - y % p
    else:
        if signY == 0:
             return - y % p
        else:
            return y
        
def elgamalEllipticEncrypt(P, QA, E, p, m):
    M = messageToPoint(m, E, p)
    if M == None:
        print("Error tolerance not big enoughe. Increase it!")
        return None
    k = randint(1, p - 1)
    kQA = doubleAndAdd(QA, k, E, p)
    C1, C2 = doubleAndAdd(P, k, E, p), addPoints(M, kQA, E, p)
    y1, y2 = C1[1], C2[1]
    C1[1], C2[1] = mSign(y1, p), mSign(y2, p)
    return [C1, C2]

def elgamalEllipticDecrypt(C, nA, E, p, K = 100):
    C1, C2 = C
    x1, b1 = C1
    x2, b2 = C2 
    y1, y2 = yCoordinate(x1, b1, E, p), yCoordinate(x2, b2, E, p)
    negC1 = [x1, - y1 % p] #-C1
    nTimesNegC1 = doubleAndAdd(negC1, nA, E, p) #-nC1
    C2[1] = y2
    Mx, My = addPoints(C2, nTimesNegC1, E, p) #C2 - nC1
    m = Mx // K
    return m

### Exercise 7

In [755]:
E = [0,7] #represents the elliptic curve y^2 = x^3 + 7
p = 2**256 - 2**32 - 977
x = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798
y = 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8
P = [x, y]
nA = 39848972070998439288129330129747770457517229308860377048978242893896467667939
QA = doubleAndAdd(P, nA, E, p)

C1 = [14597780031786705375271868769108467138015647678572751776302579165318028652678, 1]
C2 = [21210963020721916651969697565057900349212558568934755775237286996818078542813, 0]
C = [C1, C2]
m = elgamalEllipticDecrypt(C, nA, E, p)
t = intToText(m)
print("Deccrypted message:", t)

Deccrypted message: Meet me at sunrise!


### Exercise 8

### (a)

In [68]:
def reductionGauss(v1, v2):
    while True:
        x1, y1 = v1
        x2, y2 = v2
        norm1, norm2 = np.linalg.norm(v1), np.linalg.norm(v2)
        if norm2 < norm1:
            v1, v2 = v2, v1
        m = round(np.dot(v1, v2)/(np.linalg.norm(v1)**2))
        if m == 0:
            return v1
        v2 = v2 - m*v1        

### (b)

In [70]:
v1 = np.array([96, 58])
v2 = np.array([101, 61])
reductionGauss(v1, v2)

array([1, 1])

### (c)

In [72]:
v1 = np.array([66586820, 65354729])
v2 = np.array([6513996, 6393464])
reductionGauss(v1, v2)

array([ 2280, -1001])

## Exercise 9

### (a)

In [47]:
def generateTrinary(d1, d2, N):
    zeros = [0]*(N - d1 - d2)
    ones = [1]*d1
    negOnes = [-1]*d2
    l = zeros + ones + negOnes
    shuffle(l)
    return l

In [48]:
generateTrinary(4, 7, 20)

[-1, -1, -1, -1, 1, 1, 0, 0, 0, 1, 1, 0, 0, -1, 0, -1, -1, 0, 0, 0]

### (b)

In [50]:
def generateNTRU(d, p, q, N):
    f = generateTrinary(d + 1, d, N)
    g = generateTrinary(d, d, N)
    x = symbols('x')
    Poly_f, Poly_g = Poly(f, x), Poly(g, x)
    I = x ** N - 1
    Poly_I = I.as_poly()
    while True:
        try:
            Finv_p = invert(Poly_f.as_expr(), Poly_I.as_expr(), domain = GF(p))
            Finv_q = invert(Poly_f.as_expr(), Poly_I.as_expr(), domain = GF(q))
            break
        except:
            f = generateTrinary(d + 1, d, N)
            Poly_f = Poly(f, x)
    Poly_h = Poly((Finv_q * Poly_g ) % Poly_I, domain = GF(q, symmetric= False))
    return [Poly_h, Poly_f, Finv_p]


In [51]:
generateNTRU(3, 7, 11, 13)

[Poly(8*x**12 + 8*x**11 + 8*x**9 + 4*x**8 + 6*x**7 + 3*x**6 + 2*x**5 + 9*x**4 + 6*x**3 + 8*x**2 + 2*x + 2, x, modulus=11),
 Poly(x**12 + x**7 - x**6 + x**5 - x**4 - x**2 + 1, x, domain='ZZ'),
 -3*x**12 - 3*x**11 + 3*x**10 - 2*x**9 + x**8 - x**7 - 2*x**6 + 3*x**5 + x**4 - 3*x**3 - 3*x**2 + 3]

### (c)

In [53]:
def textToPoly(s, p, N):
    """
    transforms a string to a polynomial of degree at most N with coefficients modulo p
    s string,
    p prime,
    N: integer
    return: list of length N with coefficients modulo p
    """
    n = textToInt(s)
    coeffs = getExpansion(n, p)
    if len(coeffs) > N:
        print("input message too large. Try a shorter one.")
        return None
    return coeffs

def polyToText(coeff_list, p, N):
    """
    transforms a list of coefficients modulo p into a string
    coeff_list: list of coefficients modulo p,
    p: prime,
    N: prime,
    return: a string
    """
    if len(coeff_list) > N:
        print("too many coefficients. It has to be less than N")
        return None
    n = 0
    for i in range(0, len(coeff_list)):
        n += coeff_list[i]*(p**i)
    text = intToText(n)
    return text

In [54]:
s = "hej"
p = 257
N = 31
l = textToPoly(s, p, N)
l

[109, 146, 105]

In [55]:
n = polyToText(l, p, N)
n

'hej'

#### (d)

In [57]:
def encryptNTRU(d, p, q, N, m, h):
    """
    encrypts a message using NTRU algorithm
    d: pub key integer,
    p: pub key prime,
    q: pub key prime,
    N: pub key, prime,
    m: list of integers,
    h: pub key polynomial.
    return: encrypted message
    """
    d1, d2 = extendedGCD(N, p)[0], extendedGCD(p, q)[0]
    if d1 != 1:
        print("caution! gcd(N, p) must be 1")
        return None
    if d2 != 1:
        print("caution! gcd(p, q) must be 1")
        return None
    if q <= (6 * d + 1) * p:
        print("caution! q must be greater than (6d + 1)p")
        return None
    r = generateTrinary(d, d, N) #ephemeral
    x = symbols('x')
    Poly_m, Poly_r = Poly(m, x), Poly(r, x)
    Poly_m_cl = Poly(Poly_m, domain = GF(p, symmetric = True))
    Poly_I = x ** N - 1
    Poly_e = Poly((p * h * Poly_r + Poly_m_cl ) % Poly_I, domain = GF(q, symmetric= False))
    return Poly_e
    

### (e)

In [59]:
def decryptNTRU(p, q, N, f, f_p_inv, e):
    """
    decrypts a ciphertext using NTRU.
    p, q, N: primes,
    f: polynomial pub key,
    f_p_inv: inverse of f modulo p,
    e: ciphertext,
    return: list of coefficients to be decoded into a text
    """
    d1, d2 = extendedGCD(N, p)[0], extendedGCD(p, q)[0]
    if d1 != 1:
        print("caution! gcd(N, p) must be 1")
        return None
    if d2 != 1:
        print("caution! gcd(p, q) must be 1")
        return None
    if q <= (6 * d + 1) * p:
        print("caution! q must be greater than (6d + 1)p")
        return None
    x = symbols('x')
    I = x ** N - 1
    a = Poly((f * e ) % I, domain = GF(q, symmetric= True))
    b = Poly((f_p_inv * a) % I, domain = GF(p, symmetric = False))
    return b    

### (f)

In [66]:
N, p, q, d = 31, 257, 18773, 12
h, f, f_p_inv = generateNTRU(d, p, q, N)

text = "vi ses på måndag!"
m = textToPoly(text, p, N)
e = encryptNTRU(d, p, q, N, m, h)
print("Ecrypted polynomial:", e)

d = decryptNTRU(p, q, N, f, f_p_inv, e)
print("Decrypted polynomial:", d)

m2 = d.all_coeffs()
text2 = polyToText(m2, p, N)
print("Decrypted message:", text2)

Ecrypted polynomial: Poly(5930*x**30 + 8517*x**29 + 4412*x**28 + 18628*x**27 + 16357*x**26 + 12293*x**25 + 523*x**24 + 1971*x**23 + 15844*x**22 + 16768*x**21 + 17483*x**20 + 11283*x**19 + 8146*x**18 + 16375*x**17 + 13585*x**16 + 6076*x**15 + 8349*x**14 + 6796*x**13 + 4045*x**12 + 3736*x**11 + 8482*x**10 + 15639*x**9 + 12974*x**8 + 258*x**7 + 17790*x**6 + 1429*x**5 + 6288*x**4 + 14620*x**3 + 12704*x**2 + 11432*x + 1230, x, modulus=18773)
Decrypted polynomial: Poly(207*x**16 + 158*x**15 + 130*x**14 + 238*x**13 + 210*x**12 + 149*x**11 + 210*x**10 + 182*x**9 + 102*x**8 + 116*x**7 + 41*x**6 + 87*x**5 + 147*x**4 + 152*x**3 + 164*x**2 + 98*x + 31, x, modulus=257)
Decrypted message: vi ses på måndag!


## Written Part

### Exercise 1

In [64]:
E = [1, 3]
p = 7
points = generateCurve(E, p)
operations = [[i, j] for i in points for j in points]
#operations
points

['O', [4, 1], [4, 6], [5, 0], [6, 1], [6, 6]]

In [65]:
{(k[0], k[1]): addPoints(k[0], k[1], E, p) for k in operations}


TypeError: unhashable type: 'list'

In [ ]:
numbers = [1,2,3,4]
operations = [[i,j] for i in numbers for j in numbers]
operations

In [ ]:
{(k[0], k[1]): k[0] + k[1] for k in operations}